In [1]:
import pandas as pd

ruta = r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\delitos de odio\delitos de odio.csv"

df = pd.read_csv(
    ruta,
    sep=';',
    encoding='latin1',
    engine='python'
)


In [2]:
df.columns.tolist()

['Comunidades autÃ³nomas',
 'TipologÃ\xada penal',
 'Ã\x81mbito',
 'CalificaciÃ³n',
 'periodo',
 'Total']

In [3]:
df = pd.read_csv(
    ruta,
    sep=';',
    encoding='utf-8-sig',
    engine='python'
)

df.columns.tolist()

['Comunidades autónomas',
 'Tipología penal',
 'Ámbito',
 'Calificación',
 'periodo',
 'Total']

In [4]:
sorted(df['Comunidades autónomas'].unique())

['ANDALUCÍA',
 'ARAGÓN',
 'ASTURIAS (PRINCIPADO DE)',
 'BALEARS (ILLES)',
 'CANARIAS',
 'CANTABRIA',
 'CASTILLA - LA MANCHA',
 'CASTILLA Y LEÓN',
 'CATALUÑA',
 'CIUDAD AUTÓNOMA DE CEUTA',
 'CIUDAD AUTÓNOMA DE MELILLA',
 'COMUNITAT VALENCIANA',
 'DESCONOCIDO',
 'EN EL EXTRANJERO',
 'EXTREMADURA',
 'GALICIA',
 'MADRID (COMUNIDAD DE)',
 'MURCIA (REGIÓN DE)',
 'NAVARRA (COMUNIDAD FORAL DE)',
 'PAÍS VASCO',
 'RIOJA (LA)',
 'TOTAL NACIONAL']

Hay tres elementos dentro de la columna Comunidades autónomas que en realidad no son ninguna comunidad Autonónoma:
- Desconocido
- En el extranjero
- Total nacional

Procedemos a quedarnos con todos los valores de la columna menos aquello que devuelvan True a continuación, aunque guardamos los tres valores que no son Comunidades Autónomas por si más adelante pueden tener un valor analítico.

In [5]:
df_desconocido = df[df['Comunidades autónomas'] == 'DESCONOCIDO'].copy()
df_extranjero  = df[df['Comunidades autónomas'] == 'EN EL EXTRANJERO'].copy()
df_nacional    = df[df['Comunidades autónomas'] == 'TOTAL NACIONAL'].copy()


df = df[
    ~df['Comunidades autónomas'].isin([
        'DESCONOCIDO',
        'EN EL EXTRANJERO',
        'TOTAL NACIONAL'
    ])
].copy()

print(df['Comunidades autónomas'].nunique())  # debe dar 19

19


In [6]:
print("Original:", len(df_desconocido) + len(df_extranjero) + len(df_nacional) + len(df))
print("Nacional:", len(df_nacional))
print("Extranjero:", len(df_extranjero))
print("Desconocido:", len(df_desconocido))
print("Final:", len(df))

Original: 1547832
Nacional: 70356
Extranjero: 70356
Desconocido: 70356
Final: 1336764


In [7]:
df['Comunidades autónomas'].nunique()

19

Como el dataset original tiene 1.5 millones de filas, cada registro se repite para tipología penal, ámbito y calificación, nos quedamos con los totales calculados por el propio Ministerio.

In [8]:
df_limpio = df[
    (df['Calificación'] == 'TOTAL calificación') &
    (df['Tipología penal'] == 'Total general')
].copy()

# Verificar resultado
print(df_limpio.shape)
print(df_limpio[['Comunidades autónomas', 'periodo', 'Ámbito', 'Total']].head(15))

(2717, 6)
       Comunidades autónomas  periodo         Ámbito  Total
140305             ANDALUCÍA     2024  ANTIGITANISMO    4.0
140306             ANDALUCÍA     2023  ANTIGITANISMO   11.0
140307             ANDALUCÍA     2022  ANTIGITANISMO    4.0
140308             ANDALUCÍA     2021  ANTIGITANISMO    3.0
140309             ANDALUCÍA     2020  ANTIGITANISMO    2.0
140310             ANDALUCÍA     2019  ANTIGITANISMO    3.0
140311             ANDALUCÍA     2018  ANTIGITANISMO    0.0
140312             ANDALUCÍA     2017  ANTIGITANISMO    0.0
140313             ANDALUCÍA     2016  ANTIGITANISMO    0.0
140314             ANDALUCÍA     2015  ANTIGITANISMO    0.0
140315             ANDALUCÍA     2014  ANTIGITANISMO    0.0
140338             ANDALUCÍA     2024  ANTISEMITISMO    2.0
140339             ANDALUCÍA     2023  ANTISEMITISMO   11.0
140340             ANDALUCÍA     2022  ANTISEMITISMO    0.0
140341             ANDALUCÍA     2021  ANTISEMITISMO    1.0


In [9]:
# Renombrar y limpiar df_limpio
df_limpio = df_limpio.rename(columns={
    'Comunidades autónomas': 'comunidad',
    'Ámbito': 'ambito',
    'periodo': 'año',
    'Total': 'delitos'
})[['comunidad', 'año', 'ambito', 'delitos']].copy()

# Estandarizar nombres de comunidad
df_limpio['comunidad'] = (
    df_limpio['comunidad'].str.strip().str.title()
    .str.replace('Asturias (Principado De)', 'Asturias', regex=False)
    .str.replace('Balears (Illes)', 'Baleares', regex=False)
    .str.replace('Madrid (Comunidad De)', 'Madrid', regex=False)
    .str.replace('Murcia (Región De)', 'Murcia', regex=False)
    .str.replace('Navarra (Comunidad Foral De)', 'Navarra', regex=False)
    .str.replace('Ciudad Autónoma De Ceuta', 'Ceuta', regex=False)
    .str.replace('Ciudad Autónoma De Melilla', 'Melilla', regex=False)
    .str.replace('Comunitat Valenciana', 'Comunidad Valenciana', regex=False)
    .str.replace('Castilla - La Mancha', 'Castilla-La Mancha', regex=False)
    .str.replace('Rioja (La)', 'La Rioja', regex=False)
    .str.replace('Castilla Y León', 'Castilla y León', regex=False)
)

df_limpio['delitos'] = df_limpio['delitos'].astype(int)

print(sorted(df_limpio['comunidad'].unique()))
print(df_limpio.dtypes)

['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Ceuta', 'Comunidad Valenciana', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Melilla', 'Murcia', 'Navarra', 'País Vasco']
comunidad    object
año           int64
ambito       object
delitos       int64
dtype: object


In [10]:
RUTA_PROCESADOS = r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\delito_de_odio"

# Dataset 1 — para el merge final
df_limpio.to_csv(
    RUTA_PROCESADOS + r"\delitos_odio_ccaa.csv",
    index=False,
    encoding='utf-8-sig'
)

# Dataset 2 — con desglose por tipología (para análisis específicos)
df_tipologia = df[
    df['Calificación'] == 'TOTAL calificación'
].rename(columns={
    'Comunidades autónomas': 'comunidad',
    'periodo': 'año',
    'Ámbito': 'ambito',
    'Tipología penal': 'tipologia',
    'Total': 'delitos'
})[['comunidad', 'año', 'ambito', 'tipologia', 'delitos']].copy()

df_tipologia['comunidad'] = df_tipologia['comunidad'].str.strip().str.title()
df_tipologia['delitos'] = df_tipologia['delitos'].astype(int)

df_tipologia.to_csv(
    RUTA_PROCESADOS + r"\delitos_odio_tipologia.csv",
    index=False,
    encoding='utf-8-sig'
)

print("Guardado correctamente")
print(f"delitos_odio_ccaa.csv      → {len(df_limpio):,} filas")
print(f"delitos_odio_tipologia.csv → {len(df_tipologia):,} filas")

Guardado correctamente
delitos_odio_ccaa.csv      → 2,717 filas
delitos_odio_tipologia.csv → 445,588 filas


In [11]:
df_limpio[
    df_limpio['ambito'] == 'DISCRIMINACIÓN POR RAZÓN DE SEXO/GÉNERO'
].groupby('año')['delitos'].sum()

año
2014      0
2015     19
2016     41
2017     35
2018     71
2019     69
2020    101
2021    110
2022    189
2023    210
2024    188
Name: delitos, dtype: int64

In [12]:
# Guardar datasets apartados para análisis complementario
RUTA_PROCESADOS = r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\delito_de_odio"

df_desconocido.to_csv(
    RUTA_PROCESADOS + r"\delitos_desconocido.csv",
    index=False,
    encoding='utf-8-sig'
)

df_extranjero.to_csv(
    RUTA_PROCESADOS + r"\delitos_extranjero.csv",
    index=False,
    encoding='utf-8-sig'
)

df_nacional.to_csv(
    RUTA_PROCESADOS + r"\delitos_nacional.csv",
    index=False,
    encoding='utf-8-sig'
)

print(f"desconocido: {len(df_desconocido):,} filas")
print(f"extranjero:  {len(df_extranjero):,} filas")
print(f"nacional:    {len(df_nacional):,} filas")

desconocido: 70,356 filas
extranjero:  70,356 filas
nacional:    70,356 filas
